# 외부 사이트 서브검사 Excel 매크로 (Selenium 자동화)

이 노트북은 https://smart.inpsyt.co.kr/admin/at/subPsyItem/list/PAT_CO_CA/V1.1/ 에서 
서브검사상품ID별 문항, 척도, 규준, 해석 Excel 파일을 일괄 다운로드하는 자동화 매크로입니다.

**실행 순서**: 각 셀을 위에서부터 순서대로 실행하면 됩니다.
**주의**: 헤드풀 브라우저에서 로그인 창이 뜨면 수동으로 로그인을 완료하세요.

In [7]:
# ============================================================
# Step 0: 필수 라이브러리 설치
# ============================================================
# 주피터/Google Colab에서 처음 한 번만 실행하면 됩니다.

import subprocess
import sys

libraries = ["selenium", "webdriver-manager"]

for lib in libraries:
    try:
        __import__(lib)
        print(f"✓ {lib} 이미 설치됨")
    except ImportError:
        print(f"📦 {lib} 설치 중...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        print(f"✓ {lib} 설치 완료")

print("\n✓ 모든 필수 라이브러리 준비 완료")

✓ selenium 이미 설치됨
📦 webdriver-manager 설치 중...
✓ webdriver-manager 설치 완료

✓ 모든 필수 라이브러리 준비 완료


In [8]:
# ============================================================
# Step 1: 라이브러리 설치 및 임포트
# ============================================================

import os
import time
from pathlib import Path
from datetime import datetime
import json

# Selenium 라이브러리
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

print("✓ 라이브러리 임포트 완료")

✓ 라이브러리 임포트 완료


In [9]:
# ============================================================
# Step 2: 설정 및 다운로드 경로 초기화
# ============================================================

# 기본 설정
BASE_URL = "https://smart.inpsyt.co.kr"
LIST_URL = f"{BASE_URL}/admin/at/subPsyItem/list/PAT_CO_CA/V1.1/"
DOWNLOAD_DIR = Path("./macro_downloads")  # 다운로드 저장 경로
TIMEOUT = 60  # 페이지 로드 타임아웃 (초)

# 다운로드 경로 생성
DOWNLOAD_DIR.mkdir(exist_ok=True)

# 로그
print(f"✓ 기본 URL: {LIST_URL}")
print(f"✓ 다운로드 경로: {DOWNLOAD_DIR.absolute()}")

# 통계
stats = {
    "total_items": 0,
    "successful_items": 0,
    "failed_items": [],
    "file_count": {"문항": 0, "척도": 0, "규준": 0, "해석": 0},
    "start_time": datetime.now().isoformat(),
}

print(f"✓ 설정 완료")

✓ 기본 URL: https://smart.inpsyt.co.kr/admin/at/subPsyItem/list/PAT_CO_CA/V1.1/
✓ 다운로드 경로: c:\Users\user\workspace\2.0-modular\app\db\mig\macro_downloads
✓ 설정 완료


In [15]:
# ============================================================
# Step 3: Chrome WebDriver 초기화 (헤드풀 모드)
# ============================================================

# Chrome 옵션 설정
chrome_options = webdriver.ChromeOptions()
# headful 모드: 브라우저 창이 보이도록 설정
# chrome_options.add_argument('--headless')  # 주석 처리 - 헤드풀 모드로 실행

# 다운로드 경로 설정
prefs = {
    "download.default_directory": str(DOWNLOAD_DIR.absolute()),
    "download.prompt_for_download": False,
}
chrome_options.add_experimental_option("prefs", prefs)

# WebDriver 초기화
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)

print("✓ Chrome WebDriver 초기화 완료")
print("  브라우저 창이 열립니다. 로그인을 완료하세요.")

✓ Chrome WebDriver 초기화 완료
  브라우저 창이 열립니다. 로그인을 완료하세요.


In [16]:
# ============================================================
# Step 4: 사이트 접속 및 로그인 대기
# ============================================================

# 사이트 접속
driver.get(LIST_URL)
print(f"✓ {LIST_URL} 접속 완료")

# 로그인 필요 여부 확인
time.sleep(2)
page_source = driver.page_source

is_login_page = (
    "로그인" in page_source or 
    "login" in page_source or 
    "password" in page_source.lower()
)

if is_login_page:
    print("⚠ 로그인 페이지 감지됨")
    print("📍 브라우저 창에서 수동으로 로그인을 완료하세요.")
    print("   (캡차, 2FA 등 포함)")
    
    # 로그인 완료 대기 - 목록 페이지로 리다이렉트될 때까지 대기
    try:
        WebDriverWait(driver, 3600).until(
            EC.presence_of_all_elements_located((By.TAG_NAME, "table"))
        )
        print("✓ 로그인 완료 감지됨. 자동화 재개합니다.")
    except:
        print("⚠ 타임아웃 - 로그인이 완료되지 않았을 수 있습니다.")
else:
    print("✓ 이미 로그인 상태이거나 로그인 불필요")

✓ https://smart.inpsyt.co.kr/admin/at/subPsyItem/list/PAT_CO_CA/V1.1/ 접속 완료
⚠ 로그인 페이지 감지됨
📍 브라우저 창에서 수동으로 로그인을 완료하세요.
   (캡차, 2FA 등 포함)
✓ 로그인 완료 감지됨. 자동화 재개합니다.


In [17]:
# ============================================================
# Step 4-A: 윈도우 핸들 관리 및 목표 페이지 진입
# ============================================================

def switch_to_latest_window(driver):
    """
    최신 윈도우(새로 열린 탭/창)로 전환
    """
    original_window = driver.current_window_handle
    # 모든 윈도우 핸들 가져오기
    all_windows = driver.window_handles
    
    if len(all_windows) > 1:
        # 새로운 윈도우로 전환 (마지막 핸들)
        driver.switch_to.window(all_windows[-1])
        print(f"  → 새 윈도우로 전환 (총 {len(all_windows)}개 윈도우)")
        return True
    return False

def close_extra_windows(driver, keep_window_count=1):
    """
    불필요한 창 닫기. keep_window_count개의 창만 유지
    """
    try:
        all_windows = driver.window_handles
        if len(all_windows) > keep_window_count:
            # 마지막 창을 제외한 모든 창 닫기
            for window in all_windows[:-keep_window_count]:
                driver.switch_to.window(window)
                driver.close()
            # 마지막 남은 창으로 돌아가기
            driver.switch_to.window(all_windows[-1])
            print(f"  → 불필요한 창 닫음 (남은 윈도우: {keep_window_count}개)")
    except Exception as e:
        print(f"  ⚠ 창 닫기 실패: {e}")

# 원본 윈도우 핸들 저장
main_window = driver.current_window_handle
print(f"✓ 메인 윈도우 핸들 저장: {main_window}")

# 현재 페이지에서 목표 URL로 이동하는 링크 찾기
# 또는 필요시 직접 URL로 이동
time.sleep(2)

try:
    # 방법 1: 페이지 내 링크 찾기 (해당 링크가 있는 경우)
    # 예: /admin/at/subPsyItem/list/PAT_CO_CA/V1.1/ 로 가는 링크
    links = driver.find_elements(By.TAG_NAME, "a")
    target_link = None
    
    for link in links:
        href = link.get_attribute("href")
        if href and "subPsyItem" in href and "list" in href:
            print(f"  → 목표 링크 찾음: {href}")
            target_link = link
            break
    
    if target_link:
        print("  → 목표 링크 클릭...")
        target_link.click()
        time.sleep(3)
        
        # 새로운 윈도우가 열렸는지 확인
        if len(driver.window_handles) > 1:
            print(f"  ℹ 새 윈도우 감지 ({len(driver.window_handles)}개 윈도우)")
            # 새로 열린 윈도우로 자동 전환
            switch_to_latest_window(driver)
    else:
        print("  ℹ 목표 링크를 페이지에서 찾을 수 없음. 현재 URL 유지: " + driver.current_url)
        
except Exception as e:
    print(f"  ⚠ 링크 진입 중 오류: {e}")

# 모니터링을 위한 현재 윈도우 상태 출력
print(f"\n현재 상태:")
print(f"  • 활성 윈도우 수: {len(driver.window_handles)}")
print(f"  • 현재 URL: {driver.current_url}")
print(f"  • 페이지 제목: {driver.title}")
print(f"\n✓ Step 4-A 완료 (필요시 새 창이 계속 열리면 아래 셀에서 처리)")

✓ 메인 윈도우 핸들 저장: 7BC07B4561C5ACEE98227D22AA8BBA48
  → 목표 링크 찾음: https://smart.inpsyt.co.kr/admin/at/subPsyItem/list/PAT_CO_CA/V1.1/#
  → 목표 링크 클릭...

현재 상태:
  • 활성 윈도우 수: 1
  • 현재 URL: https://smart.inpsyt.co.kr/admin/at/subPsyItem/list/PAT_CO_CA/V1.1/
  • 페이지 제목: 인싸이트 관리자

✓ Step 4-A 완료 (필요시 새 창이 계속 열리면 아래 셀에서 처리)


In [ ]:
# ============================================================
# Step 4-B: 새 창 관리 유틸리티 (필요시 실행)
# ============================================================
# 🔧 실행 방법:
# - 브라우저에서 새로운 창/팝업이 계속 열리면 이 셀을 실행하세요
# - 불필요한 창들을 닫고 목표 페이지를 유지합니다

print("📋 현재 윈도우 상태 분석:")
print(f"  • 총 윈도우 수: {len(driver.window_handles)}")

for idx, handle in enumerate(driver.window_handles):
    driver.switch_to.window(handle)
    print(f"\n  [{idx + 1}] 윈도우 {idx + 1}")
    print(f"      URL: {driver.current_url}")
    print(f"      제목: {driver.title}")

# 목표 페이지 식별 (subPsyItem 또는 list 포함하는 페이지)
print(f"\n🎯 목표 페이지 찾기...")
target_window = None
for idx, handle in enumerate(driver.window_handles):
    driver.switch_to.window(handle)
    current_url = driver.current_url
    
    # 목표 페이지 조건 (둘 중 하나)
    if "subPsyItem" in current_url or "list" in current_url:
        target_window = handle
        print(f"  ✓ 목표 페이지 발견: 윈도우 {idx + 1}")
        break

if target_window:
    # 목표 페이지로 전환
    driver.switch_to.window(target_window)
    
    # 다른 모든 창 닫기
    for handle in driver.window_handles:
        if handle != target_window:
            driver.switch_to.window(handle)
            driver.close()
            print(f"  🗑 불필요한 창 닫음")
    
    # 목표 페이지로 다시 전환
    driver.switch_to.window(target_window)
    print(f"\n✓ 목표 페이지로 전환 완료")
else:
    print(f"  ⚠ 목표 페이지를 찾을 수 없음. 마지막 열린 창으로 전환합니다.")
    driver.switch_to.window(driver.window_handles[-1])

print(f"\n최종 상태:")
print(f"  • 활성 윈도우: {len(driver.window_handles)}개")
print(f"  • 현재 URL: {driver.current_url}")
print(f"  • 현재 제목: {driver.title}")
print(f"\n✓ Step 4-B 완료")

In [21]:
# ============================================================
# Step 5: 목록 테이블 파싱 및 페이지네이션
# ============================================================

def extract_items_from_page(driver):
    """
    현재 페이지에서 모든 항목 추출
    반환: [{'id': '...', 'row_element': <WebElement>}, ...]
    """
    items = []
    try:
        # 테이블 내 모든 행 찾기
        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
        print(f"  → 테이블에서 {len(rows)}개 행 찾음")
        
        for idx, row in enumerate(rows):
            try:
                # 첫 번째 셀에서 ID 추출 (또는 data-id 속성)
                cells = row.find_elements(By.TAG_NAME, "td")
                item_id = row.get_attribute("data-id")
                
                if not item_id and cells:
                    item_id = cells[0].text.strip()
                
                if item_id:
                    items.append({
                        "id": item_id,
                        "row_element": row,
                        "cells": [cell.text for cell in cells]
                    })
            except Exception as e:
                print(f"  ⚠ 행 {idx} 파싱 오류: {e}")
        
        return items
    except Exception as e:
        print(f"  ⚠ 테이블 파싱 오류: {e}")
        return []

def check_next_page(driver):
    """
    다음 페이지 버튼 존재 여부 확인
    """
    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label*='next'], button[aria-label*='next'], a:contains('다음'), a:contains('Next')")
        return next_btn
    except:
        return None

print("✓ 헬퍼 함수 정의 완료")

✓ 헬퍼 함수 정의 완료


In [22]:
# ============================================================
# Step 6: 파일 다운로드 함수 정의
# ============================================================

def download_file_from_history(driver, file_type, main_window_handle):
    """
    업로드 히스토리 보기 -> 새 창 -> 다운로드 버튼 클릭 -> 창 닫기
    file_type: '문항', '척도', '규준', '해석'
    반환: 다운로드 성공 여부 (True/False)
    """
    try:
        print(f"  [{file_type}] 업로드 히스토리 보기 시작...")
        
        # Step 1: "업로드 히스토리 보기" 버튼 찾기 및 클릭
        history_button = None
        xpath_options = [
            f"//*[contains(text(), '업로드 히스토리 보기')]",
            "//button[contains(text(), '업로드')]",
            "//a[contains(text(), '히스토리')]",
            "//button[@onclick or @data-action]",
        ]
        
        for xpath in xpath_options:
            try:
                history_button = driver.find_element(By.XPATH, xpath)
                if history_button.is_displayed():
                    print(f"    → '업로드 히스토리 보기' 버튼 찾음")
                    history_button.click()
                    time.sleep(2)
                    break
            except:
                pass
        
        if not history_button:
            print(f"    ⚠ '업로드 히스토리 보기' 버튼을 찾을 수 없음")
            return False
        
        # Step 2: 새 창이 열렸는지 확인
        all_windows = driver.window_handles
        if len(all_windows) <= 1:
            print(f"    ⚠ 새 창이 열리지 않음")
            return False
        
        # Step 3: 새 창으로 전환
        new_window = all_windows[-1]
        driver.switch_to.window(new_window)
        time.sleep(2)
        print(f"    → 새 창으로 전환 (총 {len(all_windows)}개 윈도우)")
        
        # Step 4: 최상단의 다운로드 버튼 클릭
        download_button = None
        download_xpath_options = [
            "//button[contains(text(), '다운로드')][1]",
            "//a[contains(text(), '다운로드')][1]",
            "//button[@class and contains(@class, 'download')][1]",
            "//button[1]",  # 첫 번째 버튼
            "//a[contains(@href, 'download') or contains(@href, 'file')][1]",
        ]
        
        for xpath in download_xpath_options:
            try:
                elements = driver.find_elements(By.XPATH, xpath)
                if elements:
                    download_button = elements[0]
                    if download_button.is_displayed():
                        print(f"    → 다운로드 버튼 찾음")
                        download_button.click()
                        time.sleep(2)
                        break
            except:
                pass
        
        if not download_button:
            print(f"    ⚠ 다운로드 버튼을 찾을 수 없음")
            driver.close()  # 새 창만 닫고
            driver.switch_to.window(main_window_handle)
            return False
        
        print(f"    ✓ {file_type} 파일 다운로드 완료")
        
        # Step 5: 새 창 닫기
        time.sleep(1)
        driver.close()
        print(f"    → 새 창 닫음")
        
        # Step 6: 원래 창으로 복귀
        driver.switch_to.window(main_window_handle)
        time.sleep(1)
        print(f"    → 원본 페이지로 복귀")
        
        return True
        
    except Exception as e:
        print(f"    ⚠ {file_type} 다운로드 오류: {e}")
        # 오류 발생 시 원본 창으로 복귀 시도
        try:
            if len(driver.window_handles) > 1:
                driver.close()
            driver.switch_to.window(main_window_handle)
        except:
            pass
        return False

print("✓ 파일 다운로드 함수 정의 완료")

✓ 파일 다운로드 함수 정의 완료


In [24]:
# ============================================================
# Step 7: 메인 루프 - ID별 상세 페이지 진입 및 탭별 다운로드
# ============================================================

# 탭 이름 매핑 (페이지상 탭 텍스트와 매칭해야 함)
tab_types = ["문항", "척도", "규준", "해석"]
main_window = driver.current_window_handle

print("\n🚀 매크로 시작\n")

try:
    # Step 1: 목록 페이지에서 모든 서브검사상품ID 추출
    print("=" * 60)
    print("[Step 1] 서브검사상품ID 목록 추출")
    print("=" * 60)
    
    rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
    print(f"✓ 테이블에서 {len(rows)}개 항목 찾음\n")
    
    subpsy_ids = []
    for idx, row in enumerate(rows):
        try:
            cells = row.find_elements(By.TAG_NAME, "td")
            if cells:
                subpsy_id = cells[0].text.strip()  # 첫 번째 셀이 ID
                if subpsy_id:
                    subpsy_ids.append({
                        "id": subpsy_id,
                        "element": row
                    })
                    print(f"  {idx + 1}. {subpsy_id}")
        except Exception as e:
            print(f"  ⚠ 행 {idx} 파싱 오류: {e}")
    
    if not subpsy_ids:
        print("❌ 추출된 항목이 없습니다. 종료합니다.")
    else:
        print(f"\n추출 완료: {len(subpsy_ids)}개 항목\n")
        
        # Step 2: 각 ID별로 상세 페이지 진입 후 탭별 다운로드
        for item_idx, item in enumerate(subpsy_ids, 1):
            subpsy_id = item["id"]
            print("\n" + "=" * 60)
            print(f"[{item_idx}/{len(subpsy_ids)}] 항목 처리: {subpsy_id}")
            print("=" * 60)
            
            stats["total_items"] += 1
            
            # 항목별 폴더 생성
            item_dir = DOWNLOAD_DIR / str(subpsy_id)
            item_dir.mkdir(exist_ok=True)
            print(f"저장 경로: {item_dir}\n")
            
            try:
                # Step 2-1: ID 클릭하여 상세 페이지로 진입
                print(f"[진입] {subpsy_id} 상세 페이지로 이동...")
                try:
                    id_link = item["element"].find_element(By.TAG_NAME, "a")
                    if not id_link:
                        id_link = item["element"].find_element(By.TAG_NAME, "button")
                except:
                    id_link = None
                
                if id_link:
                    id_link.click()
                    time.sleep(3)
                    print(f"  → 상세 페이지 진입 완료\n")
                else:
                    print(f"  ⚠ 진입 링크를 찾을 수 없음. 스킵합니다.\n")
                    stats["failed_items"].append(subpsy_id)
                    continue
                
                # Step 2-2: 각 탭(문항, 척도, 규준, 해석)에서 다운로드
                item_success = True
                for tab_name in tab_types:
                    print(f"\n[{tab_name}]")
                    
                    # 탭 클릭
                    try:
                        # 탭 찾기 (여러 선택자 시도)
                        tab_element = None
                        tab_xpaths = [
                            f"//div[contains(@class, 'tab') or contains(@class, 'Tab')]//span[contains(text(), '{tab_name}')]",
                            f"//button[contains(text(), '{tab_name}')]",
                            f"//a[contains(text(), '{tab_name}')]",
                            f"//*[@role='tab' and contains(text(), '{tab_name}')]",
                        ]
                        
                        for xpath in tab_xpaths:
                            try:
                                elements = driver.find_elements(By.XPATH, xpath)
                                if elements:
                                    tab_element = elements[0]
                                    break
                            except:
                                pass
                        
                        if tab_element and tab_element.is_displayed():
                            print(f"  → {tab_name} 탭 클릭")
                            tab_element.click()
                            time.sleep(2)
                        else:
                            print(f"  ⚠ {tab_name} 탭을 찾을 수 없음")
                            item_success = False
                            continue
                    except Exception as e:
                        print(f"  ⚠ {tab_name} 탭 클릭 오류: {e}")
                        item_success = False
                        continue
                    
                    # 다운로드 함수 호출
                    success = download_file_from_history(driver, tab_name, main_window)
                    if success:
                        stats["file_count"][tab_name] += 1
                        print(f"  ✓ {tab_name} 다운로드 완료")
                    else:
                        print(f"  ✗ {tab_name} 다운로드 실패")
                        item_success = False
                
                # Step 2-3: 상품 처리 완료 후 목록으로 복귀
                print(f"\n[복귀] 목록 페이지로 돌아가기...")
                driver.get(LIST_URL)
                time.sleep(2)
                print(f"  → 목록 페이지 복귀 완료")
                
                if item_success:
                    stats["successful_items"] += 1
                    print(f"\n✅ {subpsy_id} 처리 완료")
                else:
                    stats["failed_items"].append(subpsy_id)
                    print(f"\n⚠ {subpsy_id} 일부 다운로드 실패")
                    
            except Exception as e:
                print(f"\n❌ {subpsy_id} 처리 중 오류: {e}")
                stats["failed_items"].append(subpsy_id)
                # 목록으로 복귀 시도
                try:
                    driver.get(LIST_URL)
                    time.sleep(2)
                except:
                    pass

except KeyboardInterrupt:
    print("\n⚠ 사용자가 중단했습니다.")
except Exception as e:
    print(f"\n❌ 오류 발생: {e}")
finally:
    pass

print(f"\n✓ 메인 루프 완료")


🚀 매크로 시작

[Step 1] 서브검사상품ID 목록 추출
✓ 테이블에서 21개 항목 찾음

  1. PAT-2 부모양육태도검사 2판
  2. 지필형
  3. V1.1
  4. 18
  5. 17
  6. 16
  7. 15
  8. 14
  9. 13
  10. 12
  11. 11
  12. 10
  13. 9
  14. 8
  15. 7
  16. 6
  17. 5
  18. 4
  19. 3
  20. 2
  21. 1

추출 완료: 21개 항목


[1/21] 항목 처리: PAT-2 부모양육태도검사 2판
저장 경로: macro_downloads\PAT-2 부모양육태도검사 2판

[진입] PAT-2 부모양육태도검사 2판 상세 페이지로 이동...
  ⚠ 진입 링크를 찾을 수 없음. 스킵합니다.


[2/21] 항목 처리: 지필형
저장 경로: macro_downloads\지필형

[진입] 지필형 상세 페이지로 이동...
  ⚠ 진입 링크를 찾을 수 없음. 스킵합니다.


[3/21] 항목 처리: V1.1
저장 경로: macro_downloads\V1.1

[진입] V1.1 상세 페이지로 이동...
  → 상세 페이지 진입 완료


[문항]
  ⚠ 문항 탭을 찾을 수 없음

[척도]
  ⚠ 척도 탭을 찾을 수 없음

[규준]
  ⚠ 규준 탭을 찾을 수 없음

[해석]
  ⚠ 해석 탭을 찾을 수 없음

[복귀] 목록 페이지로 돌아가기...
  → 목록 페이지 복귀 완료

⚠ V1.1 일부 다운로드 실패

[4/21] 항목 처리: 18
저장 경로: macro_downloads\18

[진입] 18 상세 페이지로 이동...
  ⚠ 진입 링크를 찾을 수 없음. 스킵합니다.


[5/21] 항목 처리: 17
저장 경로: macro_downloads\17

[진입] 17 상세 페이지로 이동...
  ⚠ 진입 링크를 찾을 수 없음. 스킵합니다.


[6/21] 항목 처리: 16
저장 경로: macro_downloads\16

[진입] 16 상세 페이지로 이동...
 

In [ ]:
# ============================================================
# Step 8: 결과 리포트 생성 및 출력
# ============================================================

stats["end_time"] = datetime.now().isoformat()

# 결과 리포트 저장
report_path = DOWNLOAD_DIR / "report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print("\n" + "="*60)
print("📊 최종 결과 리포트")
print("="*60)
print(f"\n총 처리 항목: {stats['total_items']}개")
print(f"✓ 성공: {stats['successful_items']}개")
print(f"✗ 실패: {len(stats['failed_items'])}개")
if stats['failed_items']:
    print(f"  실패 항목: {', '.join(stats['failed_items'])}")

print(f"\n다운로드 파일 수:")
for file_type, count in stats['file_count'].items():
    print(f"  {file_type}: {count}개")

print(f"\n다운로드 경로: {DOWNLOAD_DIR.absolute()}")
print(f"리포트 저장: {report_path}")
print(f"\n실행 시간: {stats['start_time']} ~ {stats['end_time']}")
print("\n" + "="*60)

print(f"\n✅ 매크로 완료!")
print(f"\n💾 브라우저는 열려있습니다. 필요시 검토 후 닫으세요.")

In [ ]:
# ============================================================
# Step 9 (옵션): WebDriver 종료
# ============================================================
# 필요하면 아래 주석을 풀어서 실행하세요. (브라우저 창 닫기)

# driver.quit()
# print("✓ WebDriver 종료 및 브라우저 닫음")

# 또는 수동으로 브라우저 창을 닫고 실행하려면 위에서 아무것도 하지 말고
# 모든 셀 실행 후 브라우저 창을 손으로 닫으면 됩니다.

print("\n📝 참고: driver.quit() 주석을 풀면 브라우저가 자동 종료됩니다.")